# Info

This notebook prepares the dataset to fine-tune models for automatic speech recognition (ASR).

The dataset is prepared from raw data formatted according to `docs/schema-data_v1.1.2.json`

# Set parameters

In [ ]:
# Credentials

# env file with configs and credentials
ENV_FILE = "/home/lucar/Development/chiara-speech2text_private-v2/envs/.env"

In [ ]:
# Directories

# Local directory with raw data (downloaded from Firebase storage bucket)
DIR_DATA_RAW    = "/home/lucar/Development/chiara-speech2text_data/data_training/data_raw"

# Local directory for output dataset
DIR_DATA_OUTPUT = "/home/lucar/Development/chiara-speech2text_data/data_training/dataset"

# Local directory for hugging face dataset
DIR_DATA_HF     = "/home/lucar/Development/chiara-speech2text_data/data_training/dataset-hf"

# Output dataset in Hugging Face hub
HF_USER    = "luca-r"
HF_DATASET = "chiara-speech2text-dataset"
#HF_DATASET = "dataset-small" # for quick tests

In [ ]:
# Manually assign folders to different sets (train, test, validation)
# NOTE: Folders name defined here shall match those in the raw data directory


# - train set 
parameter_dataset_folders_train = [
]


# - test set 
parameter_dataset_folders_test = [ ]


# - "common" set
# the data from the common set will be further split into train/test/validation splits
# according to the ratios specified below

# ratios for common set splits
parameter_dataset_portion_train         = 0.8
parameter_dataset_portion_test          = 0.1
parameter_dataset_portion_validation    = 0.1


parameter_dataset_folders_common = [
    # whatsapp data
    '202311_chiara_whatsapp',
    '202312_chiara_whatsapp',
    '202401_chiara_whatsapp',
    '202402_chiara_whatsapp',
    '202403_chiara_whatsapp',
    '202404_chiara_whatsapp',
    '202405_chiara_whatsapp',
    '202406_chiara_whatsapp',
    '202407_chiara_whatsapp',
    '202408_chiara_whatsapp',
    '202409_chiara_whatsapp',
    # frontend data
    '20251020_chiara_frontend',
    '20251021_chiara_frontend',
    '20251022_chiara_frontend',
    '20251023_chiara_frontend',
    '20251024_chiara_frontend',
    '20251029_chiara_frontend',
    '20251110_chiara_frontend',
    '20251123_chiara_frontend',
    '20260104_chiara_frontend',
    '20260105_chiara_frontend',
    # dgx (keyboard) data
    '20260117_dgx',
    '20260118_dgx',
    '20260119_dgx',
    '20260121_dgx',
    '20260122_dgx',
    '20260123_dgx',
    '20260124_dgx',
    '20260125_dgx',
    '20260126_dgx',
    '20260127_dgx',
    '20260128_dgx',
    '20260129_dgx',    
    '20260130_dgx',
    '20260201_dgx',
    '20260202_dgx',
    '20260203_dgx',
]

# temp settings for quick tests
#parameter_dataset_folders_train     = ['20251020_chiara_frontend']
#parameter_dataset_folders_test      = ['20251021_chiara_frontend']
#parameter_dataset_folders_common    = ['20251022_chiara_frontend']

In [ ]:
# this is used for hf_processor
# to extract features from the audio files, and to tokenize the text labels

base_model = "openai/whisper-large-v3"
#base_model = "openai/whisper-large-v3-turbo"
#base_model = "luca-r/chiara_whisper-large-v3-turbo_20260318"
#base_model = "luca-r/chiara_whisper-large-v3-turbo_20260507"

# Initialize notebook

In [ ]:
# Install packages

%pip install --quiet pip
%pip install --quiet datasets[audio] jiwer
%pip install --quiet librosa
%pip install --quiet cupy-cuda13x
%pip install --quiet matplotlib
%pip install --quiet dotenv
%pip install --quiet ipywidgets
%pip install --quiet gsutil
%pip install --quiet transformers

In [ ]:
# Import variables from .env file
import os
from dotenv import load_dotenv

# Load the variables from the .env file into the environment
load_dotenv(ENV_FILE)

In [ ]:
# Login to Hugging Face (the processed dataset will be stored there)

from huggingface_hub import login, HfApi
from huggingface_hub.utils import HfHubHTTPError

# HF login (access models by "luca-r")

# Load Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")
#print(f"I will use the HF token: {HF_TOKEN}")

# login to HF
login(token=HF_TOKEN) # Pass the token explicitly for consistency and clarity

# Verify login by attempting to retrieve user information
try:
    HfApi().whoami()
    print("Successfully logged in to Hugging Face Hub.")
except HfHubHTTPError as e:
    raise RuntimeError(f"Hugging Face login failed: {e}. Please ensure your 'hf_token' secret is valid and you have restarted the Colab runtime after setting it.")
except Exception as e:
    raise RuntimeError(f"An unexpected error occurred during Hugging Face login verification: {e}")

# cleanup
del HF_TOKEN

In [ ]:
# Disable ipywidgets bars (they cause visualization errors with transforms library in notebooks)

import os
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import datasets
datasets.disable_progress_bars()

# Create dataset

## Show info on raw data

In [ ]:
# Get available folders

import os
print("Available folders in DIR_DATA_RAW:")
all_folders = sorted([item for item in os.listdir(DIR_DATA_RAW) 
                      if os.path.isdir(os.path.join(DIR_DATA_RAW, item))])

for i, folder in enumerate(all_folders, 1):
    folder_path = os.path.join(DIR_DATA_RAW, folder)
    wav_count = len([f for f in os.listdir(folder_path) if f.endswith('.wav')])
    print(f"{i}. {folder:30s} ({wav_count} .wav files)")

# cleanup
del i, folder, wav_count, folder_path

In [ ]:
# Show manually specified train, test, and common folders

print("\nSelected folders for 'train':")
if parameter_dataset_folders_train:
    for folder in parameter_dataset_folders_train:
        if folder in all_folders:
            print(f"  ✓ {folder}")
        else:
            print(f"  ✗ {folder} (NOT FOUND)")
else:
    print("  (none specified)")

print("\nSelected folders for 'test':")
if parameter_dataset_folders_test:
    for folder in parameter_dataset_folders_test:
        if folder in all_folders:
            print(f"  ✓ {folder}")
        else:
            print(f"  ✗ {folder} (NOT FOUND)")
else:
    print("  (none specified)")

print("\nSelected folders for 'common':")
if hasattr(parameter_dataset_folders_common, '__iter__') and parameter_dataset_folders_common:
    for folder in parameter_dataset_folders_common:
        if folder in all_folders:
            print(f"  ✓ {folder}")
        else:
            print(f"  ✗ {folder} (NOT FOUND)")
else:
    print("  (none specified)")
    
# cleanup
del folder

## Validata data splits

In [ ]:
# Validate folders selections

# Validate folders
invalid_train   = [f for f in parameter_dataset_folders_train if f not in all_folders]
invalid_test    = [f for f in parameter_dataset_folders_test  if f not in all_folders]
invalid_common  = [f for f in parameter_dataset_folders_common if f not in all_folders] if hasattr(parameter_dataset_folders_common, '__iter__') else []

# Check for overlaps between all three sets
all_folders_combined = (set(parameter_dataset_folders_train) | 
                        set(parameter_dataset_folders_test) | 
                        (set(parameter_dataset_folders_common) if hasattr(parameter_dataset_folders_common, '__iter__') else set()))
overlap_train_test = set(parameter_dataset_folders_train) & set(parameter_dataset_folders_test)
overlap_train_common = set(parameter_dataset_folders_train) & (set(parameter_dataset_folders_common) if hasattr(parameter_dataset_folders_common, '__iter__') else set())
overlap_test_common = set(parameter_dataset_folders_test) & (set(parameter_dataset_folders_common) if hasattr(parameter_dataset_folders_common, '__iter__') else set())

if invalid_train:
    print(f"❌ Invalid train folders: {invalid_train}")
if invalid_test:
    print(f"❌ Invalid test folders: {invalid_test}")
if invalid_common:
    print(f"❌ Invalid common folders: {invalid_common}")
if overlap_train_test:
    print(f"❌ Folders in both train and test: {list(overlap_train_test)}")
if overlap_train_common:
    print(f"❌ Folders in both train and common: {list(overlap_train_common)}")
if overlap_test_common:
    print(f"❌ Folders in both test and common: {list(overlap_test_common)}")

# Validate percentages
percentage_sum = parameter_dataset_portion_train + parameter_dataset_portion_test + parameter_dataset_portion_validation
if abs(percentage_sum - 1.0) > 0.001:  # Allow for floating-point rounding
    print(f"❌ Percentages do not sum to 100%: train={parameter_dataset_portion_train*100:.1f}% + test={parameter_dataset_portion_test*100:.1f}% + validation={parameter_dataset_portion_validation*100:.1f}% = {percentage_sum*100:.1f}%")
else:
    print(f"✓ Percentages are valid: train ({parameter_dataset_portion_train*100:.1f}%) + test ({parameter_dataset_portion_test*100:.1f}%) + validation ({parameter_dataset_portion_validation*100:.1f}%) = 100%")

if not invalid_train and not invalid_test and not invalid_common and not overlap_train_test and not overlap_train_common and not overlap_test_common and abs(percentage_sum - 1.0) <= 0.001:
    print("\n✓ All folders are valid and no overlaps detected")
    
    # Count total files
    train_count = 0
    test_count = 0
    common_count = 0
    
    for folder in parameter_dataset_folders_train:
        folder_path = os.path.join(DIR_DATA_RAW, folder)
        wav_count = len([f for f in os.listdir(folder_path) if f.endswith('.wav')])
        train_count += wav_count
    
    for folder in parameter_dataset_folders_test:
        folder_path = os.path.join(DIR_DATA_RAW, folder)
        wav_count = len([f for f in os.listdir(folder_path) if f.endswith('.wav')])
        test_count += wav_count
    
    for folder in parameter_dataset_folders_common:
        folder_path = os.path.join(DIR_DATA_RAW, folder)
        wav_count = len([f for f in os.listdir(folder_path) if f.endswith('.wav')])
        common_count += wav_count
    
    # Calculate split for common folder
    common_train_count = int(common_count * parameter_dataset_portion_train)
    common_test_count = int(common_count * parameter_dataset_portion_test)
    common_val_count = common_count - common_train_count - common_test_count  # Remainder goes to validation
    
    print(f"\nDirect assignment:")
    print(f"  - Train:  {train_count} files from {len(parameter_dataset_folders_train)} folders")
    print(f"  - Test:   {test_count} files from {len(parameter_dataset_folders_test)} folders")
    print(f"  - Common: {common_count} files from {len(parameter_dataset_folders_common)} folders")
    print(f"\nFrom common folder:")
    print(f"  - to Train:      {common_train_count} files ({parameter_dataset_portion_train*100:.1f}%)")
    print(f"  - to Test:       {common_test_count} files ({parameter_dataset_portion_test*100:.1f}%)")
    print(f"  - to Validation: {common_val_count} files ({parameter_dataset_portion_validation*100:.1f}%)")
    print(f"\n\nFinal dataset:")
    print(f"  - Train:      {train_count + common_train_count} files")
    print(f"  - Test:       {test_count + common_test_count} files")
    print(f"  - Validation: {common_val_count} files")
    print(f"  - Total:      {train_count + test_count + common_count} files")
else:
    if not parameter_dataset_folders_train and not parameter_dataset_folders_test and (not hasattr(parameter_dataset_folders_common, '__iter__') or not parameter_dataset_folders_common):
        print("⚠️  No folders specified yet. Please update the train_folders, test_folders, and common_folders lists above.")
    else:
        print("\n❌ Please fix the issues above before proceeding.")

# cleanup
try:
    del folder, folder_path, wav_count
    del invalid_train, invalid_test, invalid_common
    del overlap_train_test, overlap_train_common, overlap_test_common
    del train_count, test_count, common_count
    del common_train_count, common_test_count, common_val_count, percentage_sum
    del all_folders, all_folders_combined
except:
    pass

In [ ]:
# Delete old output data folder

# DIR_DATA_OUTPUT
!rm -rf {DIR_DATA_OUTPUT}
!mkdir  {DIR_DATA_OUTPUT}

In [ ]:
# Create datasets from manually specified folders with shuffled common folder
# Generate .csv and .json files


import random
import json
import csv
import shutil
import librosa
import os

# Create dataset splits
DIR_DATA_OUTPUT_TRAIN_MANIFEST = os.path.join(DIR_DATA_OUTPUT, 'train',      'train.json')
DIR_DATA_OUTPUT_TEST_MANIFEST  = os.path.join(DIR_DATA_OUTPUT, 'test',       'test.json')
DIR_DATA_OUTPUT_VAL_MANIFEST   = os.path.join(DIR_DATA_OUTPUT, 'validation', 'validation.json')

DIR_DATA_OUTPUT_TRAIN_CSV = os.path.join(DIR_DATA_OUTPUT, 'train',      'train.csv')
DIR_DATA_OUTPUT_TEST_CSV  = os.path.join(DIR_DATA_OUTPUT, 'test',       'test.csv')
DIR_DATA_OUTPUT_VAL_CSV   = os.path.join(DIR_DATA_OUTPUT, 'validation', 'validation.csv')

manifest_file_map = {
    'train':      DIR_DATA_OUTPUT_TRAIN_MANIFEST,
    'test':       DIR_DATA_OUTPUT_TEST_MANIFEST,
    'validation': DIR_DATA_OUTPUT_VAL_MANIFEST
}

csv_file_map = {
    'train':      DIR_DATA_OUTPUT_TRAIN_CSV,
    'test':       DIR_DATA_OUTPUT_TEST_CSV,
    'validation': DIR_DATA_OUTPUT_VAL_CSV
}

verbose = False

# Create output directories
!mkdir -p {DIR_DATA_OUTPUT}/train
!mkdir -p {DIR_DATA_OUTPUT}/test
!mkdir -p {DIR_DATA_OUTPUT}/validation

# Initialize CSV files with headers
for split in csv_file_map.keys():
    with open(csv_file_map[split], 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['file_name', 'transcription'])

# Process train folders (directly to train set)
train_file_count = 0

for folder_name in parameter_dataset_folders_train:
    folder_path = os.path.join(DIR_DATA_RAW, folder_name)
    wav_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.wav')])
    
    print(f"train <- {folder_name}")
    
    # Load transcripts from data.json
    transcript_map = {}
    json_path = os.path.join(folder_path, 'data.json')
    if os.path.exists(json_path):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                content = json.load(f)
                if isinstance(content, dict) and 'data' in content and isinstance(content['data'], list):
                    for entry in content['data']:
                        if isinstance(entry, dict) and 'audio_data' in entry and 'reviewer_corrected_transcript' in entry:
                            transcript_map[entry['audio_data']] = entry['reviewer_corrected_transcript']
        except Exception as e:
            print(f"  ❌ Error reading {json_path}: {e}")
    else:
        print(f"  ❌ Missing data.json in {folder_name}")
    
    for wav_file in wav_files:
        source_wav_path = os.path.join(folder_path, wav_file)
        
        # Check if transcript exists
        if wav_file not in transcript_map:
            print(f"  ❌ Missing transcript for {wav_file}")
            continue
        
        # Read transcript
        transcript = str(transcript_map[wav_file]).strip()
        if not transcript:
            print(f"  ❌ Empty transcript for {wav_file}")
            continue
        
        # Copy audio file
        target_audio_file = os.path.join(DIR_DATA_OUTPUT, 'train', f"{train_file_count}.wav")
        try:
            shutil.copyfile(source_wav_path, target_audio_file)
        except Exception as e:
            print(f"  ❌ Error copying audio file {wav_file}: {e}")
            continue
        
        # Get audio duration
        try:
            y, sr = librosa.load(source_wav_path, sr=None)
            duration = len(y) / sr
        except Exception as e:
            print(f"  ❌ Error computing duration for {wav_file}: {e}")
            continue
        
        # Write to JSON manifest
        manifest_entry = {
            "audio_filepath": target_audio_file,
            "text": transcript,
            "duration": duration
        }
        with open(manifest_file_map['train'], 'a', encoding='utf-8') as f:
            f.write(json.dumps(manifest_entry, ensure_ascii=False) + '\n')
        
        # Write to CSV
        with open(csv_file_map['train'], 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([target_audio_file, transcript])
        
        train_file_count += 1

print(f"✓ Train set (direct): {train_file_count} files")

# Process test folders (directly to test set)
test_file_count = 0

for folder_name in parameter_dataset_folders_test:
    folder_path = os.path.join(DIR_DATA_RAW, folder_name)
    wav_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.wav')])
    
    print(f"test <- {folder_name}")
    
    # Load transcripts from data.json
    transcript_map = {}
    json_path = os.path.join(folder_path, 'data.json')
    if os.path.exists(json_path):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                content = json.load(f)
                if isinstance(content, dict) and 'data' in content and isinstance(content['data'], list):
                    for entry in content['data']:
                        if isinstance(entry, dict) and 'audio_data' in entry and 'reviewer_corrected_transcript' in entry:
                            transcript_map[entry['audio_data']] = entry['reviewer_corrected_transcript']
        except Exception as e:
            print(f"  ❌ Error reading {json_path}: {e}")
    else:
        print(f"  ❌ Missing data.json in {folder_name}")
    
    for wav_file in wav_files:
        source_wav_path = os.path.join(folder_path, wav_file)
        
        # Check if transcript exists
        if wav_file not in transcript_map:
            print(f"  ❌ Missing transcript for {wav_file}")
            continue
        
        # Read transcript
        transcript = str(transcript_map[wav_file]).strip()
        if not transcript:
            print(f"  ❌ Empty transcript for {wav_file}")
            continue
        
        # Copy audio file
        target_audio_file = os.path.join(DIR_DATA_OUTPUT, 'test', f"{test_file_count}.wav")
        try:
            shutil.copyfile(source_wav_path, target_audio_file)
        except Exception as e:
            print(f"  ❌ Error copying audio file {wav_file}: {e}")
            continue
        
        # Get audio duration
        try:
            y, sr = librosa.load(source_wav_path, sr=None)
            duration = len(y) / sr
        except Exception as e:
            print(f"  ❌ Error computing duration for {wav_file}: {e}")
            continue
        
        # Write to JSON manifest
        manifest_entry = {
            "audio_filepath": target_audio_file,
            "text": transcript,
            "duration": duration
        }
        with open(manifest_file_map['test'], 'a', encoding='utf-8') as f:
            f.write(json.dumps(manifest_entry, ensure_ascii=False) + '\n')
        
        # Write to CSV
        with open(csv_file_map['test'], 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([target_audio_file, transcript])
        
        test_file_count += 1

print(f"✓ Test set (direct): {test_file_count} files")

# Process common folders: collect all files, shuffle, then split according to percentages
print(f"\nProcessing 'common' folders (shuffled split):")

common_data = []  # List of tuples: (wav_path, base_name, transcript)

for folder_name in parameter_dataset_folders_common:
    folder_path = os.path.join(DIR_DATA_RAW, folder_name)
    wav_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.wav')])
    
    print(f"  Collecting from: {folder_name}")
    
    # Load transcripts from data.json
    transcript_map = {}
    json_path = os.path.join(folder_path, 'data.json')
    if os.path.exists(json_path):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                content = json.load(f)
                if isinstance(content, dict) and 'data' in content and isinstance(content['data'], list):
                    for entry in content['data']:
                        if isinstance(entry, dict) and 'audio_data' in entry and 'reviewer_corrected_transcript' in entry:
                            transcript_map[entry['audio_data']] = entry['reviewer_corrected_transcript']
        except Exception as e:
            print(f"    ❌ Error reading {json_path}: {e}")
    else:
        print(f"    ❌ Missing data.json in {folder_name}")
    
    for wav_file in wav_files:
        base_name = os.path.splitext(wav_file)[0]
        source_wav_path = os.path.join(folder_path, wav_file)
        
        # Check if transcript exists
        if wav_file not in transcript_map:
            print(f"    ❌ Missing transcript for {wav_file}")
            continue
        
        # Read transcript to validate
        transcript = str(transcript_map[wav_file]).strip()
        if not transcript:
            print(f"    ❌ Empty transcript for {wav_file}")
            continue
        
        # Add to common_data list (source_txt_path is not needed)
        common_data.append((source_wav_path, base_name, transcript))

print(f"  Total files collected from common: {len(common_data)} files")

# Shuffle the common data to avoid temporal biases
random.seed(42)  # Set seed for reproducibility
random.shuffle(common_data)
print(f"  ✓ Data shuffled (seed=42) to avoid temporal biases")

# Split common data according to percentages
common_train_count  = int(len(common_data) * parameter_dataset_portion_train)
common_test_count   = int(len(common_data) * parameter_dataset_portion_test)
common_val_count    = len(common_data) - common_train_count - common_test_count

common_train_data   = common_data[:common_train_count]
common_test_data    = common_data[common_train_count:common_train_count + common_test_count]
common_val_data     = common_data[common_train_count + common_test_count:]

print(f"  Splitting into:")
print(f"    - Train: {common_train_count} files ({parameter_dataset_portion_train*100:.1f}%)")
print(f"    - Test: {common_test_count} files ({parameter_dataset_portion_test*100:.1f}%)")
print(f"    - Validation: {common_val_count} files ({parameter_dataset_portion_validation*100:.1f}%)")

# Process common training data
for source_wav_path, base_name, transcript in common_train_data:
    # Copy audio file
    target_audio_file = os.path.join(DIR_DATA_OUTPUT, 'train', f"{train_file_count}.wav")
    try:
        shutil.copyfile(source_wav_path, target_audio_file)
    except Exception as e:
        print(f"  ❌ Error copying audio file {base_name}.wav: {e}")
        continue
    
    # Get audio duration
    try:
        y, sr = librosa.load(source_wav_path, sr=None)
        duration = len(y) / sr
    except Exception as e:
        print(f"  ❌ Error computing duration for {base_name}.wav: {e}")
        continue
    
    # Write to JSON manifest
    manifest_entry = {
        "audio_filepath": target_audio_file,
        "text": transcript,
        "duration": duration
    }
    with open(manifest_file_map['train'], 'a', encoding='utf-8') as f:
        f.write(json.dumps(manifest_entry, ensure_ascii=False) + '\n')
    
    # Write to CSV
    with open(csv_file_map['train'], 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([target_audio_file, transcript])
    
    train_file_count += 1

# Process common test data
for source_wav_path, base_name, transcript in common_test_data:
    # Copy audio file
    target_audio_file = os.path.join(DIR_DATA_OUTPUT, 'test', f"{test_file_count}.wav")
    try:
        shutil.copyfile(source_wav_path, target_audio_file)
    except Exception as e:
        print(f"  ❌ Error copying audio file {base_name}.wav: {e}")
        continue
    
    # Get audio duration
    try:
        y, sr = librosa.load(source_wav_path, sr=None)
        duration = len(y) / sr
    except Exception as e:
        print(f"  ❌ Error computing duration for {base_name}.wav: {e}")
        continue
    
    # Write to JSON manifest
    manifest_entry = {
        "audio_filepath": target_audio_file,
        "text": transcript,
        "duration": duration
    }
    with open(manifest_file_map['test'], 'a', encoding='utf-8') as f:
        f.write(json.dumps(manifest_entry, ensure_ascii=False) + '\n')
    
    # Write to CSV
    with open(csv_file_map['test'], 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([target_audio_file, transcript])
    
    test_file_count += 1

# Process common validation data
val_file_count = 0
for source_wav_path, base_name, transcript in common_val_data:
    # Copy audio file
    target_audio_file = os.path.join(DIR_DATA_OUTPUT, 'validation', f"{val_file_count}.wav")
    try:
        shutil.copyfile(source_wav_path, target_audio_file)
    except Exception as e:
        print(f"  ❌ Error copying audio file {base_name}.wav: {e}")
        continue
    
    # Get audio duration
    try:
        y, sr = librosa.load(source_wav_path, sr=None)
        duration = len(y) / sr
    except Exception as e:
        print(f"  ❌ Error computing duration for {base_name}.wav: {e}")
        continue
    
    # Write to JSON manifest
    manifest_entry = {
        "audio_filepath": target_audio_file,
        "text": transcript,
        "duration": duration
    }
    with open(manifest_file_map['validation'], 'a', encoding='utf-8') as f:
        f.write(json.dumps(manifest_entry, ensure_ascii=False) + '\n')
    
    # Write to CSV
    with open(csv_file_map['validation'], 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([target_audio_file, transcript])
    
    val_file_count += 1

print(f"✓ Common set processed: {len(common_data)} files split and distributed")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Train set:      {train_file_count} files")
print(f"Test set:       {test_file_count} files")
print(f"Validation set: {val_file_count} files")
print(f"Total:          {train_file_count + test_file_count + val_file_count} files")
print("="*80)

# cleanup
try:
    del manifest_file_map, csv_file_map, verbose
    del folder_name, folder_path, wav_files
    del wav_file, base_name, json_path, transcript_map
    del source_wav_path, transcript
    del target_audio_file, train_file_count, test_file_count
    del writer, f
    del common_data, random
    del common_train_count, common_test_count, common_val_count
    del common_train_data, common_test_data, common_val_data
    del val_file_count
    del y, sr, duration, manifest_entry
except NameError:
    pass


In [ ]:
# cleanup

try:
    del content, entry, split
except:
    pass

## Create Hugging Face dataset

In [ ]:
# Create and fill dataset folders


from datasets import DatasetDict, Dataset
import soundfile as sf
import numpy as np


# Create dataset splits
DIR_DATA_HF_TRAIN = os.path.join(DIR_DATA_OUTPUT, 'train',      'train.csv')
DIR_DATA_HF_TEST  = os.path.join(DIR_DATA_OUTPUT, 'test',       'test.csv')
DIR_DATA_HF_VAL   = os.path.join(DIR_DATA_OUTPUT, 'validation', 'validation.csv')

# 
list_data_files = {
    "train":        DIR_DATA_HF_TRAIN,
    "test":         DIR_DATA_HF_TEST,
    "validation":   DIR_DATA_HF_VAL
}
print(list_data_files)

# Load the metadata CSV files
train_dataset   = Dataset.from_csv(DIR_DATA_HF_TRAIN)
test_dataset    = Dataset.from_csv(DIR_DATA_HF_TEST)
val_dataset     = Dataset.from_csv(DIR_DATA_HF_VAL)

# Create a DatasetDict
hf_dataset = DatasetDict({
    "train":      train_dataset,
    "test":       test_dataset,
    "validation": val_dataset
})

# Define a function to load audio and cast the 'file_name' column
def load_audio_column(batch):
    audio_path = batch["file_name"]
    try:
        audio_data, sampling_rate = sf.read(audio_path) # sf.read returns a NumPy array by default, but sometimes it might be a list depending on the file format/soundfile version
        # Ensure audio_data is a NumPy array
        if not isinstance(audio_data, np.ndarray):
            audio_data = np.array(audio_data)

        # Store relative path from DIR_DATA_OUTPUT to ensure portability across machines
        # Store as separate column (not in audio dict) because HF Audio feature strips custom fields
        relative_path = os.path.relpath(audio_path, DIR_DATA_OUTPUT)
        batch["audio"] = {"array": audio_data, "sampling_rate": sampling_rate}
        batch["audio_path"] = relative_path
    except Exception as e:
        print(f"Error loading audio file {audio_path}: {e}")
        batch["audio"] = None # Handle cases where audio loading fails
        batch["audio_path"] = None
    return batch

# Apply the function to load the audio column for all splits
num_proc = os.cpu_count()
hf_dataset = hf_dataset.map(load_audio_column, num_proc=num_proc)

print(hf_dataset)

In [ ]:
# cleanup
try:
    del train_dataset, test_dataset, val_dataset
    del num_proc
    del list_data_files
except:
    pass

### Process dataset

In [ ]:
# Create processor

# -------------------------------------------------------------------------
#                                  IMPORTANT
# -------------------------------------------------------------------------
# make sure these parameters are the !same! as those in training script

import os
import shutil
from pathlib import Path
from transformers import WhisperProcessor

# Clear transformers cache to fix potential corruption
cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
if cache_dir.exists():
    print(f"Clearing HF cache at {cache_dir}...")
    # Only clear whisper models to be safe
    for model_cache in cache_dir.glob("*whisper*"):
        try:
            shutil.rmtree(model_cache)
            print(f"  Cleared: {model_cache.name}")
        except Exception as e:
            print(f"  Warning: Could not clear {model_cache.name}: {e}")

print("\nLoading processor...")
try:
    hf_processor = WhisperProcessor.from_pretrained(
        base_model,
        language        = "it",
        task            = "transcribe",
        force_download  = True,
        cache_dir       = None
    )
    print(f"✓ Successfully loaded processor from {base_model}")
except Exception as e:
    print(f"✗ Error loading custom model: {e}")
    print("\nFalling back to official Whisper model...")
    try:
        hf_processor = WhisperProcessor.from_pretrained(
            "openai/whisper-large-v3",
            language        = "it",
            task            = "transcribe",
            force_download  = True
        )
        print("✓ Successfully loaded official Whisper processor as fallback")
        print("  Note: This is the base model without custom fine-tuning")
    except Exception as e2:
        print(f"✗ Fallback also failed: {e2}")
        raise

In [ ]:
# - Process (normalize, filter) data
# - Tokenize transcriptions
# - Extract features

# This cell takes time: ~18 minutes for ~6300 samples with device="cpu" and num_proc=None
# The cell might look stuck i.e. it keeps running but shows no output (e.g. at the "Filter" steps). if so:
# - check usage of (i) CPU and (ii) system RAM (e.g. 'btop' and 'watch -n 1 nvidia-smi' on linux)
# - if (i) usage of (1 or more) CPU core(s) is high and/or (ii) RAM usage increases, the cell is still working correctly.
# e.g.: up to ~42 GB of RAM could be allocated for ~6300 samples

from transformers.models.whisper.english_normalizer import BasicTextNormalizer

verbose = True

# - Set number of processes
# IMPORTANT: num_proc=None disables multiprocessing.
# Using multiprocessing with multiple processes in notebooks often causes timeouts and deadlocks
# even though single-process is slower, it's more reliable
num_proc = None  # Set to None to disable multiprocessing - safer in notebook environment
#num_proc = os.cpu_count()
print('processes:', num_proc)

# - Create text normalizer
# This processes text to remove capitalizations and punctuation
normalizer = BasicTextNormalizer()
print("BasicTextNormalizer initialized for pre-processing.")

# - Define dataset preparation function
iteration = 0
def prepare_dataset(batch):
    global iteration
    current_iteration = iteration
    
    # Use a local variable to avoid race conditions if num_proc > 1
    iteration += 1

    #if verbose: print(f"current_iteration: {current_iteration}")

    file_name = batch.get("file_name", "N/A")

    # --------- Normalize transcription before tokenizing
    # Ensuring transcription is a string before normalizing
    transcription_text = str(batch.get("transcription", "")).strip()
    if not transcription_text:
        print(f"Warning: Skipping sample {current_iteration} ({file_name}) due to empty or missing transcription.")
        return None

    normalized_transcription = normalizer(transcription_text)
    if not normalized_transcription: # Check again after normalization, in case normalizer returns empty string
        print(f"Warning: Skipping sample {current_iteration} ({file_name}) due to empty normalized transcription.")
        return None

    try:
        tokenized_labels = hf_processor.tokenizer(
            normalized_transcription,
            add_special_tokens          = True,     # Ensure special tokens are added for labels
            verbose                     = verbose   # Set verbose to False to avoid excessive logging from tokenizer
        ).input_ids

        if not tokenized_labels:
            print(f"Warning: Skipping sample {current_iteration} ({file_name}) due to empty tokenized labels.")
            return None

        batch["labels"] = tokenized_labels


        # --------- Extract features
        audio = batch.get("audio")

        # Check if audio is valid before processing
        if audio is None or audio.get("array") is None or len(audio["array"]) == 0:
            print(f"Warning: Skipping sample {current_iteration} ({file_name}) due to missing, invalid or empty audio data.")
            return None

        # Check sampling rate
        sampling_rate = audio.get("sampling_rate")
        if sampling_rate is None or sampling_rate <= 0:
            print(f"Warning: Skipping sample {current_iteration} ({file_name}) due to invalid sampling rate: {sampling_rate}.")
            return None

        # Feature extraction
        input_features = hf_processor.feature_extractor(
            audio["array"],
            sampling_rate=sampling_rate,
            return_attention_mask=False, # Often not needed for speech recognition, can reduce memory
            verbose=verbose # Set verbose to False to avoid excessive logging from feature_extractor
        ).input_features[0]

        if input_features is None or len(input_features) == 0:
            print(f"Warning: Skipping sample {current_iteration} ({file_name}) due to empty input features after extraction.")
            return None

        batch["input_features"] = input_features
        batch["input_length"]   = len(audio["array"]) / sampling_rate

    except Exception as e:
        print(f"Error processing sample {current_iteration} ({file_name}): {e}")
        return None # Return None if any error occurs during processing

    return batch

# - Map hugging face dataset
#hf_writer_batch_size = 1
hf_writer_batch_size = 100
hf_dataset = hf_dataset.map(  prepare_dataset,
                              writer_batch_size = hf_writer_batch_size,
                              num_proc          = num_proc,
                              desc              = "Preparing dataset",
).filter(lambda x: x is not None) # Filter out samples that returned None

# cleanup
del verbose
del num_proc
del normalizer, iteration
del hf_writer_batch_size

### Compute durations

In [ ]:
# Calculate Total Audio Duration from Prepared Dataset
# The input_length field is now available after dataset preparation

print("Computing audio duration from prepared dataset...")

# Get input_length arrays from dataset columns (fast operation)
train_input_lengths      = hf_dataset["train"]["input_length"]
test_input_lengths       = hf_dataset["test"]["input_length"]
validation_input_lengths = hf_dataset["validation"]["input_length"]

# Sum durations (input_length is in seconds)
train_duration_seconds      = sum(train_input_lengths)
test_duration_seconds       = sum(test_input_lengths)
validation_duration_seconds = sum(validation_input_lengths)

# Compute total duration in hours
metadata_total_duration_seconds            = train_duration_seconds + test_duration_seconds
metadata_data_total_duration_hours         = metadata_total_duration_seconds / 3600
metadata_data_trainset_duration_hours      = train_duration_seconds / 3600
metadata_data_testset_duration_hours       = test_duration_seconds / 3600
metadata_data_validationset_duration_hours = validation_duration_seconds / 3600

# Print results
print("AUDIO DURATION SUMMARY (after dataset preparation)")
print(f"Total duration:          {metadata_data_total_duration_hours:.2f} [hours]")
print(f"Train set duration:      {metadata_data_trainset_duration_hours:.2f} [hours]")
print(f"Test set duration:       {metadata_data_testset_duration_hours:.2f} [hours]")
print(f"Validation set duration: {metadata_data_validationset_duration_hours:.2f} [hours]")

# cleanup
del train_input_lengths, test_input_lengths, validation_input_lengths
del train_duration_seconds, test_duration_seconds, validation_duration_seconds
del metadata_total_duration_seconds

## Create dataset metadata file

In [ ]:
# Save Metadata file for dataset

import json
import os

# Create a dictionary with desired metadata
metadata = {
    "dataset_duration_total_hours":            f"{metadata_data_total_duration_hours:.2f}",
    "dataset_duration_train-set_hours":        f"{metadata_data_trainset_duration_hours:.2f}",
    "dataset_duration_test-set_hours":         f"{metadata_data_testset_duration_hours:.2f}",
    "dataset_duration_validation-set_hours":   f"{metadata_data_validationset_duration_hours:.2f}",
    "dataset_folders_train-set":               parameter_dataset_folders_train,
    "dataset_folders_test-set":                parameter_dataset_folders_test,
    "dataset_folders_common-set":              parameter_dataset_folders_common
}

# Define the path for the JSON file
metadata_file_path = os.path.join(DIR_DATA_OUTPUT, 'metadata_dataset.json')

# Save the dictionary to a JSON file
with open(metadata_file_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"Dataset metadata saved to: {metadata_file_path}")

# cleanup
del f, metadata
if 'folder' in locals(): del folder
if 'i' in locals(): del i

# Save dataset locally

In [ ]:
from datasets import load_dataset

hf_dataset.save_to_disk(f"{DIR_DATA_HF}")

# Save dataset to Hugging Face hub

## Push dataset

In [ ]:
# could be slow based on quality of internet connections and size of dataset
# e.g. 5m for ~6500 samples

from huggingface_hub import HfApi

api = HfApi()

# remove previous dataset version from hub (if any)
# delete dataset from Hugging Face Hub
api.delete_repo(
    repo_id     = HF_DATASET,
    repo_type   = "dataset",
    token       = True
)

# push dataset to Hugging Face Hub
hf_dataset.push_to_hub(
    HF_DATASET,
    private = True
)

## Push metadata

In [ ]:
# push metadata_dataset.json file to Hugging Face Hub
# Install Git LFS: https://github.com/git-lfs/git-lfs/blob/main/INSTALLING.md
api.upload_file(
    path_or_fileobj = metadata_file_path,
    path_in_repo    = 'metadata_dataset.json',
    repo_id         = f"{HF_USER}/{HF_DATASET}",
    repo_type       = 'dataset',
    commit_message  = 'Add metadata_dataset.json'
)

In [ ]:
print(f"Dataset pushed to Hugging Face Hub: {HF_USER}/{HF_DATASET}")

# Cleanup

In [ ]:
del api

In [ ]:
del hf_dataset, hf_processor

In [ ]:
del metadata_data_total_duration_hours
del metadata_data_trainset_duration_hours
del metadata_data_testset_duration_hours
del metadata_data_validationset_duration_hours

In [ ]:
del parameter_dataset_folders_common
del parameter_dataset_folders_test
del parameter_dataset_folders_train
del parameter_dataset_portion_test, parameter_dataset_portion_train, parameter_dataset_portion_validation

In [ ]:
del metadata_file_path

In [ ]:
del DIR_DATA_HF_TEST, DIR_DATA_HF_TRAIN, DIR_DATA_HF_VAL

In [ ]:
del DIR_DATA_OUTPUT_TEST_CSV, DIR_DATA_OUTPUT_TRAIN_CSV, DIR_DATA_OUTPUT_VAL_CSV
del DIR_DATA_OUTPUT_TEST_MANIFEST, DIR_DATA_OUTPUT_TRAIN_MANIFEST, DIR_DATA_OUTPUT_VAL_MANIFEST

# Next steps

Output dataset is created in `DIR_DATA_OUTPUT` and pushed to Hugging Face.

The dataset can be used to fine-tune OpenAi Whisper models and analysis

## Fine-tune OpenAi Whisper

See notebook `3-train-whisper.ipynb`

## Inspect dataset using Nvidia Nemo tool

For an intro to the tool, see: https://docs.nvidia.com/nemo-framework/user-guide/latest/nemotoolkit/tools/speech_data_explorer.html 

```bash
pip install -r /home/lucar/Development/nvidia-nemo/tools/speech_data_explorer/requirements.txt
python3 /home/lucar/Development/nvidia-nemo/tools/speech_data_explorer/data_explorer.py /home/lucar/Development/chiara-speech2text_data/data_training/dataset/test/test.json
```